<a href="https://colab.research.google.com/github/David-Z-ai/svhn-digits-generation/blob/main/svhn_vae_diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Установка и импорты

In [ ]:
!pip install -q torch torchvision matplotlib numpy tqdm tensorboard

import os
import math
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, util
from torchvision.utils import make_grid, save_image
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR

from torch.utils.tensorboard import SummaryWriter
import datetime

# Монтирование Drive и загрузка данных

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # нормализуем в диапазон [-1, 1], т.к. многие модели лучше работают с симметричным входом
])

train_dataset = datasets.SVHN(root='./data', split='train', download=True, transform=transform)
test_dataset  = datasets.SVHN(root='./data', split='test', download=True, transform=transform)

In [ ]:
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

# Exploratory data analysis

In [ ]:
def imshow(img, title=None):
    img = img / 2 + 0.5          # денормализация из [-1,1] в [0,1]
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    if title:
        plt.title(title)

images, labels = next(iter(train_loader))
plt.figure(figsize=(10, 10))
grid = make_grid(images[:16], nrow=4)
grid = grid / 2 + 0.5
plt.imshow(grid.permute(1,2,0).cpu())
plt.axis('off')
plt.show()


all_labels = []
for _, lbl in train_loader:
    all_labels.extend(lbl.numpy())
all_labels = np.array(all_labels)

plt.figure(figsize=(10, 4))
plt.hist(all_labels, bins=np.arange(-0.5, 10.5, 1), rwidth=0.8, align='mid')
plt.xticks(range(10))
plt.title('Распределение классов')
plt.xlabel('Цифра')
plt.ylabel('Количество')
plt.show()

unique, counts = np.unique(all_labels, return_counts=True)
print("\nКоличество примеров в каждом классе:")
for digit, count in zip(unique, counts):
    print(f"Цифра {digit}: {count} примеров")
print(f"Всего: {len(all_labels)} примеров")

Видно, что количество примеров в разных классах неодинаковое, поэтому проведем аугментацию данных

# Аугментация

In [ ]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class AugmentedSVHN(Dataset):
    def __init__(self, original_dataset, target_samples_per_class=None, noise_std=0.05):
        self.original = original_dataset
        self.target_samples_per_class = target_samples_per_class
        self.noise_std = noise_std

        self.class_counts = {}
        for _, label in original_dataset:
            label = int(label)
            self.class_counts[label] = self.class_counts.get(label, 0) + 1

        if target_samples_per_class is None:
            self.target_samples_per_class = max(self.class_counts.values())

        self.indices_by_class = {c: [] for c in range(10)}
        for idx, (_, label) in enumerate(original_dataset):
            self.indices_by_class[int(label)].append(idx)

        self.all_indices = []
        self.augment_flags = []
        for cls in range(10):
            current_count = len(self.indices_by_class[cls])
            needed = self.target_samples_per_class - current_count
            # Добавляем все оригинальные
            for idx in self.indices_by_class[cls]:
                self.all_indices.append(idx)
                self.augment_flags.append(False)
            # Добавляем аугментированные
            for _ in range(needed):
                orig_idx = np.random.choice(self.indices_by_class[cls])
                self.all_indices.append(orig_idx)
                self.augment_flags.append(True)
        # Перемешиваем
        shuffled = list(zip(self.all_indices, self.augment_flags))
        np.random.shuffle(shuffled)
        self.all_indices, self.augment_flags = zip(*shuffled)

    def __len__(self):
        return len(self.all_indices)

    def __getitem__(self, idx):
        orig_idx = self.all_indices[idx]
        img, label = self.original[orig_idx]
        if self.augment_flags[idx]:
            # Применяем аугментацию
            # 1. Вращение на случайный угол от -15 до 15 градусов
            angle = np.random.uniform(-15, 15)
            img = transforms.functional.rotate(img, angle, fill=0)
            # 2. Добавляем шум
            noise = torch.randn_like(img) * self.noise_std
            img = img + noise
            # 3. Нормализация
            img = torch.clamp(img, -1, 1)
        return img, label

In [ ]:
augmented_train = AugmentedSVHN(train_dataset, target_samples_per_class=13861, noise_std=0.05)

augmented_loader = DataLoader(augmented_train, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

In [ ]:
print("Распределение классов после аугментации:")
counts = [0]*10
for _, lbl in augmented_train:
    counts[lbl] += 1
for i, cnt in enumerate(counts):
    print(f"Цифра {i}: {cnt} примеров")
print(f"Всего: {sum(counts)} примеров")

plt.figure(figsize=(10, 4))
plt.hist([lbl for _, lbl in augmented_train], bins=np.arange(-0.5, 10.5, 1), rwidth=0.8, align='mid')
plt.xticks(range(10))
plt.title('Распределение классов после аугментации')
plt.xlabel('Цифра')
plt.ylabel('Количество')
plt.show()


def visualize_augmentations(dataset, num_samples=3, num_augments=3):
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    fig, axes = plt.subplots(num_samples, num_augments+1, figsize=(3*(num_augments+1), 3*num_samples))
    for i, idx in enumerate(indices):
        img, label = dataset[idx]
        # Оригинал
        axes[i, 0].imshow(img.permute(1,2,0).cpu() / 2 + 0.5)
        axes[i, 0].set_title(f"Оригинал {label}")
        axes[i, 0].axis('off')
        # Генерируем аугментированные версии (те же операции, что в AugmentedSVHN)
        for aug in range(num_augments):
            aug_img = img.clone()
            angle = np.random.uniform(-15, 15)
            aug_img = transforms.functional.rotate(aug_img, angle, fill=0)
            noise = torch.randn_like(aug_img) * 0.05
            aug_img = aug_img + noise
            aug_img = torch.clamp(aug_img, -1, 1)
            axes[i, aug+1].imshow(aug_img.permute(1,2,0).cpu() / 2 + 0.5)
            axes[i, aug+1].set_title(f"Аугм {aug+1}")
            axes[i, aug+1].axis('off')
    plt.tight_layout()
    plt.show()

visualize_augmentations(train_dataset, num_samples=3, num_augments=3)

Проверяем устройство

In [ ]:
if 'device' not in locals():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Используется устройство: {device}")

# Variational autoencoder

VAE (Variational Autoencoder) - генеративная модель, которая сжимает входное изображение в вероятностное латентное пространство (mu и logvar) и учится восстанавливать исходную картинку из случайной точки в этом пространстве. Энкодер свёрточный с шагом 2 (размерность уменьшается в 8 раз), декодер: симметричная транспонированная свёртка с tanh на выходе. Репараметризация позволяет обучаться через градиентный спуск.

In [ ]:
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, stride=2, padding=1)
        self.fc_mu = nn.Linear(64 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(64 * 4 * 4, latent_dim)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

In [ ]:
class VAEDecoder(nn.Module):
    def __init__(self, latent_dim=10):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 64 * 4 * 4)
        self.deconv1 = nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1)
        self.deconv2 = nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1)
        self.deconv3 = nn.ConvTranspose2d(16, 3, 3, stride=2, padding=1, output_padding=1)

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 64, 4, 4)
        x = F.relu(self.deconv1(x))
        x = F.relu(self.deconv2(x))
        x = torch.tanh(self.deconv3(x))
        return x

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=10):
        super().__init__()
        self.encoder = VAEEncoder(latent_dim)
        self.decoder = VAEDecoder(latent_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

In [ ]:
def vae_loss(recon, x, mu, logvar):
    recon_loss = F.mse_loss(recon, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss, recon_loss, kl_loss

def train_vae(vae, train_loader, epochs=50, lr=1e-3, device='cuda'):
    vae.to(device)
    optimizer = optim.Adam(vae.parameters(), lr=lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    train_losses = []
    recon_losses = []
    kl_losses = []
    mae_history = []

    for epoch in range(epochs):
        vae.train()
        total_loss = 0
        total_recon = 0
        total_kl = 0
        total_mae = 0
        count = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for batch_idx, (data, _) in enumerate(pbar):
            data = data.to(device)
            optimizer.zero_grad()
            recon, mu, logvar = vae(data)
            loss, recon_loss, kl_loss = vae_loss(recon, data, mu, logvar)
            loss.backward()
            optimizer.step()

            batch_size = data.size(0)
            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()
            mae = torch.mean(torch.abs(recon - data)).item() * batch_size
            total_mae += mae
            count += batch_size

            pbar.set_postfix({'loss': loss.item() / batch_size,
                              'recon': recon_loss.item() / batch_size,
                              'kl': kl_loss.item() / batch_size,
                              'mae': mae / batch_size})

        avg_loss = total_loss / count
        avg_recon = total_recon / count
        avg_kl = total_kl / count
        avg_mae = total_mae / count
        train_losses.append(avg_loss)
        recon_losses.append(avg_recon)
        kl_losses.append(avg_kl)
        mae_history.append(avg_mae)

        scheduler.step()
        print(f'Epoch {epoch+1}: Loss = {avg_loss:.4f}, Recon = {avg_recon:.4f}, KL = {avg_kl:.4f}, MAE = {avg_mae:.4f}')

    return train_losses, recon_losses, kl_losses, mae_history

In [ ]:
vae = VAE(latent_dim=10)
vae_losses, recon_losses, kl_losses, mae_history = train_vae(vae, train_loader, epochs=30, lr=1e-3, device=device)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(vae_losses)
plt.title('Общая потеря VAE')
plt.xlabel('Эпоха')
plt.ylabel('Loss')

plt.subplot(1, 3, 2)
plt.plot(recon_losses, label='Reconstruction loss')
plt.plot(kl_losses, label='KL loss')
plt.legend()
plt.title('Компоненты потери')

plt.subplot(1, 3, 3)
plt.plot(mae_history)
plt.title('MAE реконструкции')
plt.xlabel('Эпоха')
plt.ylabel('MAE')

plt.tight_layout()
plt.show()

In [ ]:
vae.eval()
test_iter = iter(test_loader)
test_images, test_labels = next(test_iter)
test_images = test_images.to(device)

with torch.no_grad():
    recon, _, _ = vae(test_images)

def denorm(img):
    return img / 2 + 0.5

n = 10
plt.figure(figsize=(20, 4))
for i in range(n):
    ax = plt.subplot(2, n, i+1)
    plt.imshow(denorm(test_images[i].cpu()).permute(1,2,0))
    plt.axis('off')
    ax = plt.subplot(2, n, i+1+n)
    plt.imshow(denorm(recon[i].cpu()).permute(1,2,0))
    plt.axis('off')
plt.suptitle('VAE реконструкции (верхний ряд - оригиналы, нижний - реконструкции)')
plt.show()

In [ ]:
with torch.no_grad():
    z = torch.randn(20, 10).to(device)
    generated = vae.decoder(z)
    generated = denorm(generated).cpu()

plt.figure(figsize=(10, 4))
grid = make_grid(generated, nrow=10, padding=2)
plt.imshow(grid.permute(1,2,0))
plt.axis('off')
plt.title('Генерация VAE из случайного шума')
plt.show()

Сохраняем модель.

In [ ]:
# Путь к папке
save_dir = '/content/drive/MyDrive/pet_project_models'
os.makedirs(save_dir, exist_ok=True)
print(f"Папка для сохранения: {save_dir}")

In [ ]:
# Сохранение VAE
vae_name = f"vae_ep30_bs{batch_size}_latent10"
vae_path = os.path.join(save_dir, f"{vae_name}.pth")
torch.save(vae.state_dict(), vae_path)

vae_config = {
    'latent_dim': 10,
    'image_channels': 3,
    'epochs_trained': 30,
    'batch_size': batch_size,
    'loss': 'MSE+KL',
}
torch.save(vae_config, os.path.join(save_dir, f"{vae_name}_config.pth"))

vae_history = {
    'loss': vae_losses,
    'recon_loss': recon_losses,
    'kl_loss': kl_losses,
    'mae': mae_history,
}
torch.save(vae_history, os.path.join(save_dir, f"{vae_name}_history.pth"))

# Denoising Diffusion Probabilistic Model

DDPM (Denoising Diffusion Probabilistic Model) постепенно зашумляет изображение за 1000 шагов и учится восстанавливать его обратно, предсказывая добавленный шум. Модель условная: UNet получает эмбеддинг времени (синусоидальный) и эмбеддинг цифры (10 классов). NoiseScheduler предвычисляет коэффициенты для прямого и обратного процессов (линейное расписание betas).

In [ ]:
class NoiseScheduler:
    def __init__(self, num_timesteps=1000, beta_start=1e-4, beta_end=0.02, schedule='linear'):
        self.num_timesteps = num_timesteps
        if schedule == 'linear':
            self.betas = torch.linspace(beta_start, beta_end, num_timesteps)
        elif schedule == 'cosine':
            # косинусное расписание
            steps = num_timesteps + 1
            x = torch.linspace(0, num_timesteps, steps)
            alphas_cumprod = torch.cos(((x / num_timesteps) + 0.008) / (1 + 0.008) * math.pi / 2) ** 2
            alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
            betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
            betas = torch.clip(betas, 0.0001, 0.9999)
            self.betas = betas
        else:
            raise ValueError(f'Unknown schedule {schedule}')

        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.)

        # Коэффициенты для forward diffusion
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)

        # Коэффициенты для обратного процесса (posterior)
        self.posterior_variance = self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)
        self.posterior_mean_coef1 = self.betas * torch.sqrt(self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)
        self.posterior_mean_coef2 = (1. - self.alphas_cumprod_prev) * torch.sqrt(self.alphas) / (1. - self.alphas_cumprod)

    def q_sample(self, x0, t, noise=None):
        """Добавить шум к x0 на шаге t"""
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_alpha_cumprod_t = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_cumprod_t = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
        return sqrt_alpha_cumprod_t * x0 + sqrt_one_minus_alpha_cumprod_t * noise, noise

    def q_posterior(self, x0, xt, t):
        """Вычислить среднее и дисперсию q(x_{t-1}|x_t, x0)"""
        posterior_mean = (
            self.posterior_mean_coef1[t].view(-1,1,1,1) * x0 +
            self.posterior_mean_coef2[t].view(-1,1,1,1) * xt
        )
        posterior_variance = self.posterior_variance[t]
        return posterior_mean, posterior_variance

    def p_sample(self, model, xt, t, classes, clip_denoised=True):
        """Один шаг обратного процесса: предсказать x_{t-1}"""
        t_batch = torch.full((xt.shape[0],), t, device=xt.device, dtype=torch.long)
        predicted_noise = model(xt, t_batch, classes)
        # Предсказать x0 из xt и предсказанного шума
        alpha_cumprod_t = self.alphas_cumprod[t].view(-1,1,1,1)
        sqrt_alpha_cumprod_t = self.sqrt_alphas_cumprod[t].view(-1,1,1,1)
        sqrt_one_minus_alpha_cumprod_t = self.sqrt_one_minus_alphas_cumprod[t].view(-1,1,1,1)
        pred_x0 = (xt - sqrt_one_minus_alpha_cumprod_t * predicted_noise) / sqrt_alpha_cumprod_t
        if clip_denoised:
            pred_x0 = torch.clamp(pred_x0, -1., 1.)

        # Вычислить среднее и дисперсию q(x_{t-1}|x_t, x0)
        mean, variance = self.q_posterior(pred_x0, xt, t)
        if t == 0:
            return mean
        else:
            noise = torch.randn_like(xt)
            return mean + torch.sqrt(variance).view(-1,1,1,1) * noise

    def sample(self, model, num_samples, classes, img_shape=(3,32,32), device='cuda', clip_denoised=True):
        """Генерация изображений из случайного шума"""
        model.eval()
        with torch.no_grad():
            xt = torch.randn((num_samples,) + img_shape, device=device)
            for t in reversed(range(self.num_timesteps)):
                xt = self.p_sample(model, xt, t, classes, clip_denoised)
            return xt

    def to(self, device):
        """Переместить все предвычисленные тензоры на указанное устройство"""
        self.betas = self.betas.to(device)
        self.alphas = self.alphas.to(device)
        self.alphas_cumprod = self.alphas_cumprod.to(device)
        self.alphas_cumprod_prev = self.alphas_cumprod_prev.to(device)
        self.sqrt_alphas_cumprod = self.sqrt_alphas_cumprod.to(device)
        self.sqrt_one_minus_alphas_cumprod = self.sqrt_one_minus_alphas_cumprod.to(device)
        self.posterior_variance = self.posterior_variance.to(device)
        self.posterior_mean_coef1 = self.posterior_mean_coef1.to(device)
        self.posterior_mean_coef2 = self.posterior_mean_coef2.to(device)
        return self

Синусоидальные эмбединги вычисляем согласно формуле:

$\text{PE}(t, 2i) = \sin\left( t / 10000^{2i/d} \right), \quad \text{PE}(t, 2i+1) = \cos\left( t / 10000^{2i/d} \right)$

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

Self-Attention для пространственных признаков (аналог non-local блока). Входная карта признаков `x` (B×C×H×W) через свёртки 1×1 превращается в query (Q) и key (K) с пониженной размерностью `C/reduction` и value (V) с полной размерностью `C`. Attention-веса вычисляются как `softmax(Q·K^T / sqrt(C'))`, затем применяются к V. Результат добавляется к исходному входу (residual connection). Это позволяет модели учитывать глобальные зависимости между пикселями.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.channels = channels
        self.reduction = reduction
        self.q_conv = nn.Conv2d(channels, channels // reduction, 1)
        self.k_conv = nn.Conv2d(channels, channels // reduction, 1)
        self.v_conv = nn.Conv2d(channels, channels, 1)
        self.out_conv = nn.Conv2d(channels, channels, 1)
        self.scale = (channels // reduction) ** -0.5

    def forward(self, x):
        B, C, H, W = x.shape
        # q, k, v: [B, C', H, W] -> [B, H*W, C'] / [B, C', H*W] / [B, H*W, C]
        q = self.q_conv(x).view(B, -1, H*W).permute(0,2,1)  # [B, H*W, C']
        k = self.k_conv(x).view(B, -1, H*W)                 # [B, C', H*W]
        v = self.v_conv(x).view(B, -1, H*W).permute(0,2,1) # [B, H*W, C]
        # attention: softmax( q @ k^T / sqrt(C') )  -> [B, H*W, H*W]
        attn = torch.bmm(q, k) * self.scale
        attn = F.softmax(attn, dim=-1)
        out = torch.bmm(attn, v).permute(0,2,1).view(B, C, H, W)
        out = self.out_conv(out)
        return x + out

Базовый блок UNet с поддержкой даунсемплинга (`down`), апсемплинга (`up`) и residual connection через attention. Содержит два свёрточных слоя 3×3 с BatchNorm и ReLU, в которые аддитивно встраиваются эмбеддинги времени и класса (после первого свёрточного слоя). При включённом `use_attention` применяется SelfAttention. В конце, если задан `up` или `down`, размер карты признаков меняется вдвое (транспонированная свёртка или свёртка с шагом 2).

In [ ]:
class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, class_emb_dim, up=False, down=False, use_attention=False):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch) if time_emb_dim else None
        self.class_mlp = nn.Linear(class_emb_dim, out_ch) if class_emb_dim else None
        if up:
            self.conv1 = nn.Conv2d(2*in_ch, out_ch, 3, padding=1)
            self.transform = nn.ConvTranspose2d(out_ch, out_ch, 4, 2, 1)
        elif down:
            self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
            self.transform = nn.Conv2d(out_ch, out_ch, 4, 2, 1)
        else:
            self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
            self.transform = nn.Identity()

        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bnorm1 = nn.BatchNorm2d(out_ch)
        self.bnorm2 = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU()
        self.use_attention = use_attention
        if use_attention:
            self.attention = SelfAttention(out_ch)
        else:
            self.attention = None

    def forward(self, x, t=None, class_labels=None):
        h = self.act(self.bnorm1(self.conv1(x)))
        if self.time_mlp and t is not None:
            time_emb = self.time_mlp(t)
            h = h + time_emb[:, :, None, None]
        if self.class_mlp and class_labels is not None:
            class_emb = self.class_mlp(class_labels)
            h = h + class_emb[:, :, None, None]
        h = self.act(self.bnorm2(self.conv2(h)))
        if self.attention is not None:
            h = self.attention(h)
        return self.transform(h)

UNet для условной диффузионной модели. Энкодер последовательно уменьшает пространственное разрешение (64 → 128 → 256 каналов), декодер восстанавливает его обратно с skip-соединениями через `cat`. На уровнях `down2` и `up2` добавлены SelfAttention для захвата глобальных зависимостей. Эмбеддинги времени (через синусоидальное кодирование + MLP) и класса (Embedding) подаются в каждый блок, интегрируясь после первого свёрточного слоя.

In [ ]:
class SimpleUNet(nn.Module):
    def __init__(self, image_channels=3, time_emb_dim=128, class_emb_dim=10, num_classes=10):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.ReLU()
        )
        self.class_mlp = nn.Embedding(num_classes, class_emb_dim)

        # Энкодер
        self.down1 = Block(image_channels, 64, time_emb_dim, class_emb_dim, down=True)
        self.down2 = Block(64, 128, time_emb_dim, class_emb_dim, down=True, use_attention=True)
        self.down3 = Block(128, 256, time_emb_dim, class_emb_dim, down=True)

        # Bottleneck
        self.bottleneck = Block(256, 256, time_emb_dim, class_emb_dim, use_attention=True)

        # Декодер
        self.up3 = Block(256, 128, time_emb_dim, class_emb_dim, up=True)
        self.up2 = Block(128, 64, time_emb_dim, class_emb_dim, up=True, use_attention=True)
        self.up1 = Block(64, 64, time_emb_dim, class_emb_dim, up=True)

        self.out = nn.Conv2d(64, image_channels, 1)

    def forward(self, x, t, class_labels):
        # Временной эмбединг
        t_emb = self.time_mlp(t)
        # Эмбединг класса
        c_emb = self.class_mlp(class_labels)

        d1 = self.down1(x, t_emb, c_emb)
        d2 = self.down2(d1, t_emb, c_emb)
        d3 = self.down3(d2, t_emb, c_emb)

        b = self.bottleneck(d3, t_emb, c_emb)

        u3 = self.up3(torch.cat([b, d3], dim=1), t_emb, c_emb)
        u2 = self.up2(torch.cat([u3, d2], dim=1), t_emb, c_emb)
        u1 = self.up1(torch.cat([u2, d1], dim=1), t_emb, c_emb)

        return self.out(u1)

Модель учится предсказывать добавленный шум для случайного временного шага t (MSE loss). На каждом шаге оптимизатор Adam с косинусным планировщиком, дополнительно логируется MAE. Каждые 5 эпох генерируется 20 примеров (по 2 на каждый класс) для визуального контроля качества.

In [ ]:
def train_diffusion(model, noise_scheduler, train_loader, epochs=100, lr=1e-4, device='cuda'):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    losses = []
    mae_errors = []   # MAE между предсказанным и истинным шумом (для визуализации)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_mae = 0
        count = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for batch_idx, (images, labels) in enumerate(pbar):
            images = images.to(device)
            labels = labels.to(device)
            batch_size = images.size(0)

            # Случайные временные шаги
            t = torch.randint(0, noise_scheduler.num_timesteps, (batch_size,), device=device)

            # Добавить шум к изображениям
            noise = torch.randn_like(images)
            x_t, _ = noise_scheduler.q_sample(images, t, noise)

            # Предсказать шум
            predicted_noise = model(x_t, t, labels)

            # Потеря: MSE между предсказанным и истинным шумом
            loss = F.mse_loss(predicted_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch_size
            # MAE между предсказанным и истинным шумом
            mae = torch.mean(torch.abs(predicted_noise - noise)).item() * batch_size
            total_mae += mae
            count += batch_size

            pbar.set_postfix({'loss': loss.item(), 'mae': mae / batch_size})

        avg_loss = total_loss / count
        avg_mae = total_mae / count
        losses.append(avg_loss)
        mae_errors.append(avg_mae)

        scheduler.step()
        print(f'Epoch {epoch+1}: Loss = {avg_loss:.4f}, MAE = {avg_mae:.4f}')

        # Каждые 5 эпох показываем сгенерированные примеры
        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                # Генерируем по одному примеру для каждого класса
                class_labels = torch.arange(10, device=device).repeat(2)
                generated = noise_scheduler.sample(model, 20, class_labels, img_shape=(3,32,32), device=device)
                generated = generated / 2 + 0.5
                grid = make_grid(generated, nrow=10, padding=2)
                plt.figure(figsize=(12, 3))
                plt.imshow(grid.permute(1,2,0).cpu())
                plt.axis('off')
                plt.title(f'Сгенерированные изображения на эпохе {epoch+1}')
                plt.show()

    return losses, mae_errors

# Инициализация
noise_scheduler = NoiseScheduler(num_timesteps=1000, beta_start=1e-4, beta_end=0.02, schedule='linear')
noise_scheduler.to(device)
model = SimpleUNet(image_channels=3, time_emb_dim=128, class_emb_dim=10, num_classes=10)

# Обучение
diffusion_losses, diffusion_mae = train_diffusion(model, noise_scheduler, train_loader, epochs=120, lr=1e-4, device=device)


plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(diffusion_losses)
plt.title('Потеря (MSE предсказанного шума)')
plt.xlabel('Эпоха')
plt.ylabel('Loss')

plt.subplot(1, 2, 2)
plt.plot(diffusion_mae)
plt.title('MAE предсказанного шума')
plt.xlabel('Эпоха')
plt.ylabel('MAE')
plt.tight_layout()
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    class_labels = torch.arange(10, device=device).repeat(2)
    generated = noise_scheduler.sample(model, 20, class_labels, img_shape=(3,32,32), device=device)
    generated = generated / 2 + 0.5  # denormalize

# Визуализация
plt.figure(figsize=(15, 5))
grid = make_grid(generated, nrow=10, padding=2)
plt.imshow(grid.permute(1,2,0).cpu())
plt.axis('off')
plt.title('Сгенерированные изображения условной диффузионной моделью (два примера на класс)')
plt.show()

Сохраняем модель.

In [ ]:
import os

save_dir = '/content/drive/MyDrive/pet_project_models'
os.makedirs(save_dir, exist_ok=True)
print(f"Папка для сохранения: {save_dir}")

In [ ]:
# Формируем имя файла, отражающее основные параметры модели
model_name = f"unet_ep120_bs{batch_size}_t1000"
model_path = os.path.join(save_dir, f"{model_name}.pth")
torch.save(model.state_dict(), model_path)
print(f"Модель сохранена в {model_path}")

# Сохраняем полную конфигурацию модели
model_config = {
    'image_channels': 3,
    'num_classes': 10,
    'time_emb_dim': 128,
    'class_emb_dim': 10,
    'attention_layers': ['down2', 'up2'],  # где были attention блоки
    'norm_type': 'GroupNorm',
    'use_ema': True,
    'epochs_trained': 120,
    'batch_size': batch_size,
    'optimizer': 'Adam',
    'lr': 1e-4,
    'scheduler': 'CosineAnnealingLR',
}
config_path = os.path.join(save_dir, f"{model_name}_config.pth")
torch.save(model_config, config_path)

# Сохраняем параметры шумового планировщика
scheduler_config = {
    'num_timesteps': noise_scheduler.num_timesteps,
    'beta_start': 1e-4,
    'beta_end': 0.02,
}
scheduler_config_path = os.path.join(save_dir, f"{model_name}_scheduler.pth")
torch.save(scheduler_config, scheduler_config_path)



history = {
    'losses': diffusion_losses,
    'mae': diffusion_mae,
}
history_path = os.path.join(save_dir, f"{model_name}_history.pth")
torch.save(history, history_path)

print("Все файлы сохранены.")